# Домашнє завдання №7 — RAG для фінансових звітів компаній

**Тема:** Retrieval-Augmented Generation (RAG) системи.

**Мета:** побудувати RAG-пайплайн, який знаходить релевантні фінансові дані компаній та відповідає українською мовою з точними цифрами.

**Стек:** `langchain`, `langchain-openai`, `langchain-chroma`, `RecursiveCharacterTextSplitter`, `Chroma`, `OpenAIEmbeddings`.

## Крок 1. Встановлення бібліотек

Встановлюємо залежності для побудови RAG-пайплайна: **LangChain** для композиції компонентів, **langchain-openai** для моделей OpenAI, **langchain-chroma** для векторного сховища Chroma та **langchain-text-splitters** для поділу тексту.

Команда `%pip` встановлює пакети в середовище поточного ноутбука. Якщо після інсталяції Colab попросить перезапустити сеанс, це потрібно зробити перед наступними кроками.

In [2]:
# Встановлення залежностей
%pip install -U langchain langchain-openai langchain-chroma langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 577.2/577.2 kB 17.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 89.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 104.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.1 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installatio

## Крок 2. Безпечне налаштування OpenAI API-ключа

Спочатку перевіряємо змінну середовища `OPENAI_API_KEY`, потім секрети Google Colab і, якщо ключа немає, просимо ввести його приховано через `getpass`.

Ключ використовується для викликів embedding-моделі та LLM. **Не вставляємо ключ безпосередньо в ноутбук** і не публікуємо його у виводі.

In [4]:
import os
from getpass import getpass

# Безпечне завантаження API ключа: env -> Colab Secrets -> ручне введення
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    try:
        from google.colab import userdata
        colab_key = userdata.get("OPENAI_API_KEY")
        if colab_key:
            api_key = colab_key
    except Exception:
        pass

if not api_key:
    api_key = getpass("Введіть OPENAI_API_KEY: ").strip()

if not api_key:
    raise ValueError("OPENAI_API_KEY не задано.")

os.environ["OPENAI_API_KEY"] = api_key
print("OPENAI_API_KEY: OK")

OPENAI_API_KEY: OK


## Крок 3. Імпорт компонентів RAG

Імпортуємо класи для кожного етапу: `RecursiveCharacterTextSplitter` для **chunking**, `OpenAIEmbeddings` і `Chroma` для **indexing/retrieval**, `ChatPromptTemplate`, `ChatOpenAI` і LCEL-компоненти для **generation**.

На цьому етапі моделі й сховище ще не створюються — ми лише підключаємо потрібні Python-класи.

In [5]:
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## Крок 4. Вхідні фінансові дані та логічне чанкування

Формуємо текстові дані чотирьох вигаданих українських компаній за 2024 рік. Далі розділяємо текст за порожніми рядками (`separators=["\n\n"]`): **одна компанія = один chunk**.

Параметри `chunk_size=300` та `chunk_overlap=0` залишають короткі записи компаній окремими фрагментами без дублювання. У `metadata["company"]` записуємо назву компанії. `assert` перевіряє головну вимогу завдання: **рівно 4 chunks**.

In [6]:
financial_text = """ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу."""

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=300,
    chunk_overlap=0,
)

chunks = text_splitter.create_documents([financial_text])

for doc in chunks:
    first_line = doc.page_content.split("\n", 1)[0]
    company_name = first_line.split(" за 2024 рік", 1)[0].strip()
    doc.metadata = {"company": company_name}

assert len(chunks) == 4, f"Очікується 4 чанки, але отримано {len(chunks)}"

print(f"Кількість чанків: {len(chunks)}")
for i, doc in enumerate(chunks, 1):
    print("=" * 70)
    print(f"Чанк {i}: {doc.metadata['company']}")
    print(doc.page_content)


Кількість чанків: 4
Чанк 1: ТехНова Інк
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.
Чанк 2: ГрінЕнерджі Солюшнс
ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.
Чанк 3: РітейлМакс Груп
РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.
Чанк 4: ФінТех Дайнамікс
ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Праців

## Крок 5. Створення embeddings і векторного сховища

`OpenAIEmbeddings(model="text-embedding-3-small")` перетворює кожен chunk на числовий вектор. `Chroma.from_documents(...)` індексує ці вектори разом із текстом та метаданими.

Колекція `finansovi_zvity_kompaniy` має змістовну назву. Для записів задаємо стабільні ідентифікатори та перевіряємо, що в сховищі **4 документи**. Цей етап викликає API для створення ембедингів.

In [7]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    ids=[
        "tehnova_2024",
        "greenenergy_2024",
        "retailmax_2024",
        "fintech_dynamics_2024",
    ],
    collection_name="finansovi_zvity_kompaniy",
    collection_metadata={
        "description": "Фінансові звіти українських компаній за 2024 рік"
    },
)

collection_count = vectorstore._collection.count()
print(f"Документів у колекції: {collection_count}")
assert collection_count == 4, f"У векторному сховищі має бути 4 документи, а не {collection_count}"

Документів у колекції: 4


## Крок 6. Налаштування й перевірка retriever

Перетворюємо векторне сховище на retriever із `search_type="similarity"` та `k=2`: для кожного запитання система повертає **два найбільш схожі chunks**.

Тестовим запитом перевіряємо, скільки документів знайдено і які компанії зазначені в їхніх метаданих. Важливо: similarity search повертає найближчі записи, навіть коли точного збігу немає.

In [8]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

demo_question = "Яка виручка у компанії ТехНова Інк за 2024 рік?"
retrieved_docs = retriever.invoke(demo_question)

print(f"Retriever повернув {len(retrieved_docs)} документи")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"{i}. {doc.metadata.get('company', 'Невідома компанія')}")

Retriever повернув 2 документи
1. ТехНова Інк
2. ФінТех Дайнамікс


## Крок 7. Prompt, LLM та повний RAG-ланцюг

Налаштовуємо інструкцію відповідати **українською**, використовувати **лише знайдений контекст** і не вигадувати відсутні фінансові показники.

`format_docs()` об'єднує тексти знайдених фрагментів, а `extract_target()` формує текстову назву об'єкта для повідомлення про відсутні дані. Через **LCEL** об'єднуємо `retriever → prompt → LLM → StrOutputParser` в один `rag_chain`.

`temperature=0` зменшує випадковість генерації, але не гарантує правильність відповіді.

In [9]:
prompt = ChatPromptTemplate.from_template(
    """Ти фінансовий асистент, який відповідає українською мовою.
Використовуй ТІЛЬКИ наданий контекст.
Надавай коротку точну відповідь з конкретними цифрами та одиницями, якщо вони є.
Якщо у контексті немає потрібної компанії або показника, відповідай рівно так:
"На основі наданого контексту, інформація про {target} відсутня."

Питання: {question}
Контекст:
{context}

Відповідь:"""
)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    """Об'єднати тексти знайдених документів для передачі в контекст LLM.

    Args:
        docs: Послідовність LangChain Document, отриманих retriever.

    Returns:
        str: Тексти документів, розділені двома символами нового рядка.
    """
    return "\n\n".join(doc.page_content for doc in docs)

def extract_target(question: str) -> str:
    """Виділити орієнтовний об'єкт запиту для повідомлення про відсутні дані.

    Args:
        question: Оригінальне запитання користувача українською мовою.

    Returns:
        str: Частина запитання після першого « у » або повний запит без «?».

    Notes:
        Це проста евристика, а не повноцінне розпізнавання назви компанії.
    """
    q = question.strip().rstrip("?")
    if " у " in q:
        return q.split(" у ", 1)[1]
    return q

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "target": lambda q: extract_target(q),
    }
    | prompt
    | llm
    | StrOutputParser()
)


## Крок 8. Перевірка на шести запитаннях

Виконуємо всі тестові запитання з умови домашнього завдання: п'ять про компанії, наявні в документах, і одне про **«ЕкоПакування Солюшнс»**, якої немає у вхідних даних.

Після кожного виклику `rag_chain.invoke(...)` виводимо запитання й відповідь. Перевіряємо не тільки цифри, а й **правильну компанію, знак від'ємних значень і одиниці вимірювання**.

In [10]:
questions = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
    "Який чистий прибуток у ЕкоПакування Солюшнс?",
    "Який churn rate у компанії Cordial?"
]

print("=" * 70)
for i, question in enumerate(questions, 1):
    print(f"\nПитання {i}: {question}")
    answer = rag_chain.invoke(question)
    print(f"Відповідь: {answer}")
    print("-" * 70)



Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка у компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------

Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: 5100 осіб.
----------------------------------------------------------------------

Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE у РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------

Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
----------------------------------------------------------------------

Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
Відповідь: Маржа EBITDA у ФінТех Дайнамікс становить мінус 8 відсотків.
----------------------------------------------------------------------

Питання 6: Який чистий прибуток у ЕкоПакування Солюшнс?
Відповідь: На основі наданого конт

## Крок 9. Попередня автоматизована перевірка відповідей

Для шести результатів виконуємо просту перевірку наявності очікуваних текстових фрагментів і виводимо `PASS` / `FAIL`.

**Обмеження:** пошук підрядка — це лише швидкий smoke-check. Наприклад, наявність `"8"` ще не доводить, що система правильно відповіла **«мінус 8 відсотків»**. Остаточно звіряємо відповіді з умовою завдання.

In [11]:
expected_facts = [
    "450",
    "5100",
    "9.3",
    "Львові",
    "8",
    "відсутня"
]

answers = [rag_chain.invoke(q) for q in questions]

for i, (answer, expected) in enumerate(
    zip(answers, expected_facts), start=1
):
    passed = expected.lower() in answer.lower()
    print(f"Test {i}: {'PASS' if passed else 'FAIL'}")

Test 1: PASS
Test 2: PASS
Test 3: PASS
Test 4: PASS
Test 5: PASS
Test 6: PASS


## Висновки

У межах домашнього завдання реалізовано та перевірено базову RAG-систему для пошуку й аналізу фінансових показників компаній.

- **Chunking:** вихідний текст розбито на 4 логічні чанки — по одному для кожної компанії. Це дозволило зберегти фінансові показники кожної компанії в межах окремого документа.
- **Vector Store:** створено колекцію `finansovi_zvity_kompaniy` у Chroma, що містить 4 документи, проіндексовані за допомогою `OpenAIEmbeddings`.
- **Retrieval:** retriever налаштовано з параметром `k=2`, відповідно до вимог завдання.
- **Generation:** RAG-ланцюг успішно обробив україномовні запити та сформував відповіді на основі знайденого контексту.
- **Перевірка результатів:** виконано 7 тестових запитів замість 6 обов'язкових. Усі 5 запитань щодо наявних компаній отримали правильні відповіді з конкретними фінансовими показниками або фактичними даними.
- **Обробка відсутньої інформації:** система коректно повідомила про відсутність даних для двох компаній, яких немає у вихідних документах: `ЕкоПакування Солюшнс` та `Cordial`. Це підтвердило, що під час перевірки система не вигадала фінансові показники для невідомих компаній.

**Результат:** усі 7 тестових запитів оброблено коректно (7/7). Реалізована RAG-система відповідає функціональним критеріям домашнього завдання на перевіреному наборі даних.